# Fine-tuning DistilBERT Base Model for Extractive Question Answering with Stanford Question Answering Dataset (SQuAD) 

## 1. Introduction

This notebook provides...

Core objectives...



## 2. Initial Setup

### 1.1 GPU Check

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### 1.2 Library Installation

First, we need to install the necessary Python libraries. We'll use the -q flag for a quiet installation to keep the output clean.

- transformers: Provides the LLM models and tokenizers (e.g., our base model, TinyLlama).
- ...

In [1]:
pip install -q transformers datasets evaluate protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requi

# 3. Loading the dataset, tokenizer and model

The dataset that is used the most as an academic benchmark for extractive question answering is SQuAD, so that’s the one we’ll use here. There is also a harder SQuAD v2 benchmark, which includes questions that don’t have an answer. As long as your own dataset contains a column for contexts, a column for questions, and a column for answers, you should be able to adapt the steps below.

As usual, we can download and cache the dataset in just one step thanks to load_dataset().

The context and question fields are very straightforward to use. The answers field is a bit trickier as it comports a dictionary with two fields that are both lists. This is the format that will be expected by the squad metric during evaluation; if you are using your own data, you don’t necessarily need to worry about putting the answers in the same format. The text field is rather obvious, and the answer_start field contains the starting character index of each answer in the context.

During training, there is only one possible answer.

For evaluation, however, there are several possible answers for each sample, which may be the same or different. Some of the questions have several possible answers, and this script will compare a predicted answer to all the acceptable answers and take the best score.

The `load_dataset` function from the `datasets` library downloads and caches the data for us.

Data Structure: The dataset returns a `Dataset` object, which is similar to a Python dictionary. Each entry has a key (e.g., 'train') and a value containing the data. The data itself is structured with a single column named text that contains the full instruction prompt and response.

In [3]:
from datasets import load_dataset

squad = load_dataset("squad")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

We need to convert the text in the input into IDs the model can make sense of, using a tokenizer:

As mentioned previously, we’ll be fine-tuning a BERT model, but you can use any other model type as long as it has a fast tokenizer implemented. We can pass to our tokenizer the question and the context together, and it will properly insert the special tokens to form a sentence like this: [CLS] question [SEP] context [SEP]. The labels will then be the index of the tokens starting and ending the answer, and the model will be tasked to predicted one start and end logit per token in the input. 

**Tokenizer**: The tokenizer converts our text prompts into a format the model can understand (tokens). We load the tokenizer that corresponds to our chosen model. We set `padding_side="right"` to prevent issues with certain model architectures during open-ended generation. In open-ended generation tasks (for example, when you provide a prompt and the model continues the text), if padding is added to the left (i.e., at the beginning of the input), some models may get confused and start generating text based on the padding tokens.


In [4]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

**Model**: Initialize DistilBERT with Question Answering head (linear layer for start/end logits).

In [10]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint).to(device)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


X

## 4. Preprocessing

### 4.1. Preprocessing the Training Dataset

Let’s start with preprocessing the training data. The hard part will be to generate labels for the question’s answer, which will be the start and end positions of the tokens corresponding to the answer inside the context.

In this case the context is not too long, but some of the examples in the dataset have very long contexts that will exceed the maximum length we set (which is 384 in this case). We will deal with long contexts by creating several training features from one sample of our dataset, with a sliding window between them.

The dataset provides us with the start character of the answer in the context, and by adding the length of the answer, we can find the end character in the context. To map those to token indices, we will need to use the offset mappings we studied in Chapter 6. We can have our tokenizer return these by passing along return_offsets_mapping=True:

As we can see, we get back the usual input IDs, token type IDs, and attention mask, as well as the offset mapping we required and an extra key, overflow_to_sample_mapping. The corresponding value will be of use to us when we tokenize several texts at the same time (which we should do to benefit from the fact that our tokenizer is backed by Rust). Since one sample can give several features, it maps each feature to the example it originated from.


This information will be useful to map each feature we get to its corresponding label. As mentioned earlier, those labels are:

- (0, 0) if the answer is not in the corresponding span of the context
- (start_position, end_position) if the answer is in the corresponding span of the context, with start_position being the index of the token (in the input IDs) at the start of the answer and end_position being the index of the token (in the input IDs) where the answer ends

To determine which of these is the case and, if relevant, the positions of the tokens, we first find the indices that start and end the context in the input IDs. We could use the token type IDs to do this, but since those do not necessarily exist for all models (DistilBERT does not require them, for instance), we’ll instead use the sequence_ids() method of the BatchEncoding our tokenizer returns.

Once we have those token indices, we look at the corresponding offsets, which are tuples of two integers representing the span of characters inside the original context. We can thus detect if the chunk of the context in this feature starts after the answer or ends before the answer begins (in which case the label is (0, 0)). If that’s not the case, we loop to find the first and last token of the answer:

We’ll pad every feature to the maximum length we set, as most of the contexts will be long (and the corresponding samples will be split into several features), so there is no real benefit to applying dynamic padding here. 

In [ ]:
max_length = 384
stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],

        max_length=max_length, # to set the maximum length
        truncation="only_second", #  to truncate the context (which is in the second position) when the question with its context is too long
        stride=stride, # to set the number of overlapping tokens between two successive chunks
        return_overflowing_tokens=True, # to let the tokenizer know we want the overflowing tokens
        return_offsets_mapping=True, # to return the offset mappings
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

Note that we defined two constants to determine the maximum length used as well as the length of the sliding window, and that we added a tiny bit of cleanup before tokenizing: some of the questions in the SQuAD dataset have extra spaces at the beginning and the end that don’t add anything (and take up space when being tokenized if you use a model like RoBERTa), so we removed those extra spaces.

To apply this function to the whole training set, we use the Dataset.map() method with the batched=True flag. It’s necessary here as we are changing the length of the dataset (since one example can give several training features):

In [6]:
train_dataset = squad["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=squad["train"].column_names,
)

len(squad["train"]), len(train_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

(87599, 88729)

As we can see, the preprocessing added roughly 1,000 features. Our training set is now ready to be used — let’s dig into the preprocessing of the validation set!

### 4.2. Preprocessing the Validation Dataset

Preprocessing the validation data will be slightly easier as we don’t need to generate labels (unless we want to compute a validation loss, but that number won’t really help us understand how good the model is). The real joy will be to interpret the predictions of the model into spans of the original context. For this, we will just need to store both the offset mappings and some way to match each created feature to the original example it comes from. Since there is an ID column in the original dataset, we’ll use that ID.

The only thing we’ll add here is a tiny bit of cleanup of the offset mappings. They will contain offsets for the question and the context, but once we’re in the post-processing stage we won’t have any way to know which part of the input IDs corresponded to the context and which part was the question (the sequence_ids() method we used is available for the output of the tokenizer only). So, we’ll set the offsets corresponding to the question to None:

In [ ]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length, # to set the maximum length
        truncation="only_second", #  to truncate the context (which is in the second position) when the question with its context is too long
        stride=stride, # to set the number of overlapping tokens between two successive chunks
        return_overflowing_tokens=True, # to let the tokenizer know we want the overflowing tokens
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

We can apply this function on the whole validation dataset like before:

In [8]:
validation_dataset = squad["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=squad["validation"].column_names,
)
len(squad["validation"]), len(validation_dataset)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

(10570, 10822)

In this case we’ve only added a couple of hundred samples, so it appears the contexts in the validation dataset are a bit shorter.

Now that we have preprocessed all the data, we can get to the training.

## 5. Fine-tuning the model with the Trainer API

### 5.1. Arguments definition

Once this is done, we can define our TrainingArguments. As we said when we defined our function to compute the metric, we won’t be able to have a regular evaluation loop because of the signature of the compute_metrics() function. We could write our own subclass of Trainer to do this (an approach you can find in the question answering example script), but that’s a bit too long for this section. Instead, we will only evaluate the model at the end of training here and show you how to do a regular evaluation in “A custom training loop” below.

This is really where the Trainer API shows its limits and the Accelerate library shines: customizing the class to a specific use case can be painful, but tweaking a fully exposed training loop is easy.

We’ve seen most of these before: we set some hyperparameters (like the learning rate, the number of epochs we train for, and some weight decay) and indicate that we want to save the model at the end of every epoch, skip evaluation, and upload our results to the Model Hub. We also enable mixed-precision training with fp16=True, as it can speed up the training nicely on a recent GPU.

This object defines all the hyperparameters for the training process:

- Arguments

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-squad",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=1, # @todo increase
    weight_decay=0.01,
    fp16=True,
    report_to="none",
)

### 5.2. Trainer creation

Then we just pass everything to the Trainer.  This orchestrates the entire training process.

- Parameter explanation


Remember that one *step* is one mini training cycle using a batch of examples.  During a single step, the model:

1. Processes a batch of input data

2. Performs a forward pass to generate predictions

3. Computes the loss by comparing predictions to the ground truth

4. Performs a backward pass to compute gradients and updates the model's weights


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model, # Base model 
    args=args, # Training configuration
    train_dataset=train_dataset, # Training data
    eval_dataset=validation_dataset, # Validation data
    processing_class=tokenizer,
)

Step,Training Loss
1,5.958000
100,4.550000
200,3.710600
300,3.090300
400,2.621400
500,2.498800
600,2.415800
700,2.343400
800,2.270100
900,2.282200


TrainOutput(global_step=11092, training_loss=1.5391867679247906, metrics={'train_runtime': 1794.6106, 'train_samples_per_second': 49.442, 'train_steps_per_second': 6.181, 'total_flos': 8694539184554496.0, 'train_loss': 1.5391867679247906, 'epoch': 1.0})

### 5.3. Model Training

Finally we launch training. This may take time depending on the model size and hardware.

In [ ]:
trainer.train()

Note that while the training happens, each time the model is saved (here, every epoch) it is uploaded to the Hub in the background. This way, you will be able to to resume your training on another machine if necessary. The whole training takes a while (a little over an hour on a Titan RTX), so you can grab a coffee or reread some of the parts of the course that you’ve found more challenging while it proceeds. Also note that as soon as the first epoch is finished, you will see some weights uploaded to the Hub and you can start playing with your model on its page.

## 6. Evaluation

The training code for this example will look a lot like the code in the previous sections — the hardest thing will be to write the compute_metrics() function. Since we padded all the samples to the maximum length we set, there is no data collator to define, so this metric computation is really the only thing we have to worry about. The difficult part will be to post-process the model predictions into spans of text in the original examples. 

The model will output logits for the start and end positions of the answer in the input IDs, as we saw during our exploration of the question-answering pipeline. The post-processing step will be similar to what we did there. 

- We masked the start and end logits corresponding to tokens outside of the context.
- We then converted the start and end logits into probabilities using a softmax.
- We attributed a score to each (start_token, end_token) pair by taking the product of the corresponding two probabilities.
- We looked for the pair with the maximum score that yielded a valid answer (e.g., a start_token lower than end_token).

Here we will change this process slightly because we don’t need to compute actual scores (just the predicted answer). This means we can skip the softmax step. To go faster, we also won’t score all the possible (start_token, end_token) pairs, but only the ones corresponding to the highest n_best logits (with n_best=20). Since we will skip the softmax, those scores will be logit scores, and will be obtained by taking the sum of the start and end logits. 

As we said before, we’ll look at the logit scores for the n_best start logits and end logits, excluding positions that give:

- An answer that wouldn’t be inside the context
- An answer with negative length
- An answer that is too long

Once we have all the scored possible answers for one example, we just pick the one with the best logit score. 

The final format of the predicted answers is the one that will be expected by the metric we will use. As usual, we can load it with the help of the Evaluate library  This metric expects the predicted answers in a list of dictionaries with one key for the ID of the example and one key for the predicted text, and the theoretical answers in a list of dictionaries with one key for the ID of the example and one key for the possible answers.

Now let’s put everything we just did in a compute_metrics() function that we will use in the Trainer. Normally, that compute_metrics() function only receives a tuple eval_preds with logits and labels. Here we will need a bit more, as we have to look in the dataset of features for the offset and in the dataset of examples for the original contexts, so we won’t be able to use this function to get regular evaluation results during training. We will only use it at the end of training to check the results.

In [ ]:
from tqdm.auto import tqdm
import collections
import numpy as np
import evaluate

# Constants for answer extraction
n_best = 20
max_answer_length = 30

# Load SQuAD metric for evaluation
metric = evaluate.load("squad")

def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

2025-11-24 19:55:55.079416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764014155.264088      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764014155.318721      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Once the training is complete, we can finally evaluate our model (and pray we didn’t spend all that compute time on nothing). The predict() method of the Trainer will return a tuple where the first elements will be the predictions of the model (here a pair with the start and end logits). We send this to our compute_metrics() function:

In [13]:
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, squad["validation"])

  0%|          | 0/10570 [00:00<?, ?it/s]

{'exact_match': 73.8221381267739, 'f1': 82.73834276879933}

## 7. Performance After Fine-Tuning Check

We will ask the fine-tuned model the exact same question as the base model and see if its response has improved. The fine-tuned model should  provide a much more direct and accurate answer, following the instruction format it learned from the SQuAD dataset.

We will use Hugging Face’s `pipeline` utility to easily create text-generation pipelines  for both the base model and the fine-tuned model. This allows us to generate text  in a simple and consistent way.

In [ ]:
from transformers import pipeline

# Createas pipeline for base model
base_qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert-base-cased"
)

# Creates a pipeline for fine-tuned model
tuned_qa_pipeline = pipeline(
    task="question-answering",
    model=trainer.model,
    tokenizer=tokenizer
)

# Test cases
test_cases = [
    {
        "context": """
        Retrieval-Augmented Generation (RAG), introduced by Lewis et al. in 2020, is designed to provide Large Language Models (LLMs) with additional information from an external knowledge source.
        The main purpose of RAG is to generate more accurate and contextual answers while reducing hallucinations.
        In RAG, factual knowledge is separated from the LLMs reasoning capability and stored in an external knowledge source.
        This technique is analogous to taking an open-book exam: “RAG is to LLMs what an open-book exam is to humans”.
        """,
        "question": "What is the primary function of Retrieval-Augmented Generation (RAG) in LLM applications?"
    },
    {
        "context": """
        GPT-3 is an autoregressive language model released by OpenAI in 2020.
        This model is characterized by its scale, which includes 96 Transformer decoder layers.
        The model possesses 175 billion parameters and was trained using 300 billion tokens from several sources.
        Furthermore, GPT-3 has a 2048-token-long context window, which represents the amount of text the model can consider when generating a response.
        """,
        "question": "How many parameters does the GPT-3 model have?"
    }, 
    {
        "context": """
        The transformer architecture was introduced by Vaswani et al. in the paper “Attention Is All You Need” (2017).
        The architecture eliminates Recurrent Neural Networks (RNNs) and uses only attention mechanisms.
        While the quality of MT was improved, the architecture especially drastically improves the training speed because it deals with sequential data in parallel, allowing the use of GPUs/TPUs.
        Language Models like Gemini and ChatGPT are based on transformers.
        """,
        "question": "Which research paper introduced the Transformer architecture and in what year?"
    },
    {
        "context": """
        Long short-term memory (LSTM) models were developed to address long-term dependencies, solving issues found in traditional Recurrent Neural Networks (RNNs) that often exhibit short-term memory.
        LSTMs introduce the cell state, which is key to their functionality.
        The cell state acts as the long-term memory of the LSTM, designed to carry information across many time steps while undergoing minimal modifications.
        In contrast, the hidden state represents the short-term, immediate output, used for predictions and communication with other layers.
        """,
        "question": "What role does the cell state play within the Long short-term memory (LSTM) architecture?"
    },
    {
        "context": """
        Word embeddings are designed not only to reflect similarity but also to preserve semantic and syntactic relationships.
        These relationships are observed in the difference vectors between words.
        This property allows for analogies or queries to be expressed as equations
        Furthermore, these sophisticated relationships can be leveraged in other applications, such as assisting with machine translation.
        """,
        "question": "How do word embeddings encode semantic and syntactic relationships between words?"
    }
]

# Run comparison
for i, test_case in enumerate(test_cases, 1):
    context = test_case["context"].strip()
    question = test_case["question"]

    # Get prediction from base model
    base_result = base_qa_pipeline(question=question, context=context)
    base_answer = base_result["answer"]
    base_score = base_result["score"]

    # Get prediction from fine-tuned model
    tuned_result = tuned_qa_pipeline(question=question, context=context)
    tuned_answer = tuned_result["answer"]
    tuned_score = tuned_result["score"]

    print(f"Test case #{i}\n")
    print(f"Context: {context}\n")
    print(f"Question: {question}\n")

    print(f"Base model:\n")
    print(f"- Answer: {base_answer}\n")
    print(f"- Confidence: {base_score:.4f}\n")

    print(f"Fine-tuned model:\n")
    print(f"- Answer: {tuned_answer}\n")
    print(f"- Confidence: {tuned_score:.4f}\n")

## 8. Conclusion
In this notebook, we have successfully:

- 